In [1]:
pip install pandas pyxlsb openpyxl


Note: you may need to restart the kernel to use updated packages.


In [19]:
# =============================================
# INSTALL FIRST (run this in terminal once)
# pip install pandas pyxlsb
# =============================================

import pandas as pd
import pyxlsb
import re
import warnings
from datetime import datetime, timedelta
warnings.filterwarnings('ignore')


# ─────────────────────────────────────────
# 1. LOAD THE FILE
# ─────────────────────────────────────────

print("Loading... this will take 8-10 minutes")

df_main = pd.read_excel(
    "Keonjhar_household_survey__Odia_-_all_versions_-_labels_-_2026-05-04-09-15-09.xlsb",
    engine    = 'pyxlsb',
    sheet_name = 'Keonjhar household survey ( ...',
    dtype     = object       # ← this line stops pandas from dropping values
)

df_mem = pd.read_excel(
    "Keonjhar_household_survey__Odia_-_all_versions_-_labels_-_2026-05-04-09-15-09.xlsb",
    engine     = 'pyxlsb',
    sheet_name = 'member_details',
    dtype      = object      # ← same here
)

print(f"Sheet 1 loaded : {df_main.shape[0]} rows, {df_main.shape[1]} columns")
print(f"Sheet 2 loaded : {df_mem.shape[0]} rows, {df_mem.shape[1]} columns")

# quick check
print()
print("start column raw values :")
print(df_main['start'].head(5).tolist())

Loading... this will take 8-10 minutes
Sheet 1 loaded : 67307 rows, 202 columns
Sheet 2 loaded : 238471 rows, 73 columns

start column raw values :
[46113.25980086806, 46113.26781034722, 46113.2732629051, 46113.27861564815, 46113.28822680555]


In [3]:
# ─────────────────────────────────────────
# 2. REMOVE COLUMNS THAT ARE FULLY EMPTY
# ─────────────────────────────────────────
# Some columns have NO data at all — not a single value
# There is no point keeping them
# Example: show_block_code, _notes, _tags — all blank

df_main.dropna(axis=1, how='all', inplace=True)
df_mem.dropna(axis=1, how='all',  inplace=True)

print(f"\nAfter removing empty columns:")
print(f"Sheet 1 : {df_main.shape[1]} columns remaining")
print(f"Sheet 2 : {df_mem.shape[1]} columns remaining")


After removing empty columns:
Sheet 1 : 187 columns remaining
Sheet 2 : 64 columns remaining


In [4]:
# ─────────────────────────────────────────
# 3. REMOVE KOBO SYSTEM COLUMNS
# ─────────────────────────────────────────
# KoBoToolbox adds its own technical columns to every export
# These are NOT survey answers — they are system information
# Power BI does not need them for any chart or report

kobo_columns_to_remove = [
    '_id',
    '_submission_time',
    'gps',
    'duration',
    '_status',
    '__version__',
    'meta/rootUuid',
    '_submission__id',
    '_submission__status',
    '_submission___version__',
    '_submission_meta/rootUuid',
    '_parent_table_name',
    '_submission__submission_time'
]

for col in kobo_columns_to_remove:
    if col in df_main.columns:
        df_main.drop(columns=[col], inplace=True)
    if col in df_mem.columns:
        df_mem.drop(columns=[col], inplace=True)

print("\nKobo system columns removed")


Kobo system columns removed


In [5]:
# ─────────────────────────────────────────
# 4. REMOVE SUB-COLUMNS OF MULTI-SELECT QUESTIONS
# ─────────────────────────────────────────
# In KoBoToolbox, when a question has multiple checkboxes
# it creates one column per checkbox like:
#   Reason_for_dropout/Financial burden   → 0 or 1
#   Reason_for_dropout/Distance           → 0 or 1
# These columns have a "/" in the name
# The PARENT column already has the full answer
# So these sub-columns are not needed

df_main.drop(columns=[c for c in df_main.columns if '/' in str(c)], inplace=True)
df_mem.drop(columns=[c for c in df_mem.columns  if '/' in str(c)], inplace=True)

print(f"\nAfter removing sub-columns:")
print(f"Sheet 1 : {df_main.shape[1]} columns remaining")
print(f"Sheet 2 : {df_mem.shape[1]} columns remaining")


After removing sub-columns:
Sheet 1 : 138 columns remaining
Sheet 2 : 43 columns remaining


In [20]:
# ─────────────────────────────────────────
# 5. FIX THE DATE COLUMNS
# ─────────────────────────────────────────

def convert_excel_date(number):
    try:
        return datetime(1899, 12, 30) + timedelta(days=float(number))
    except:
        return None

df_main['start'] = df_main['start'].apply(convert_excel_date)
df_main['start'] = pd.to_datetime(df_main['start']).dt.strftime('%Y-%m-%d %H:%M:%S')

df_main['end'] = df_main['end'].apply(convert_excel_date)
df_main['end'] = pd.to_datetime(df_main['end']).dt.strftime('%Y-%m-%d %H:%M:%S')

df_main['interview_date']         = pd.to_datetime(df_main['start']).dt.strftime('%Y-%m-%d')
df_main['interview_duration_min'] = (
    (pd.to_datetime(df_main['end']) - pd.to_datetime(df_main['start']))
    .dt.total_seconds() / 60
).round(1)

print("start sample :", df_main['start'].head(3).tolist())
print("end sample   :", df_main['end'].head(3).tolist())

start sample : ['2026-04-01 06:14:06', '2026-04-01 06:25:38', '2026-04-01 06:33:29']
end sample   : ['2026-04-01 06:20:52', '2026-04-01 06:32:54', '2026-04-01 06:40:49']


In [13]:
# ─────────────────────────────────────────
# 6. REMOVE EXTRA SPACES FROM TEXT COLUMNS
# ─────────────────────────────────────────

for col in df_main.columns:
    if df_main[col].dtype == object:
        try:
            df_main[col] = df_main[col].str.strip()
        except:
            pass   # skip columns that are not actually text

for col in df_mem.columns:
    if df_mem[col].dtype == object:
        try:
            df_mem[col] = df_mem[col].str.strip()
        except:
            pass   # skip columns that are not actually text

print("Extra spaces removed from all text columns")

Extra spaces removed from all text columns


In [14]:
# ─────────────────────────────────────────
# 7. CLEAN UP COLUMN NAMES
# ─────────────────────────────────────────
# Some column names have spaces, capital letters, special characters
# Power BI DAX formulas break with messy column names
# We convert everything to simple lowercase with underscores
# Example : "Block Code" becomes "block_code"

def make_clean_name(name):
    name = str(name).strip().lower()
    name = re.sub(r'[\s\-]+', '_', name)   # spaces and dashes → underscore
    name = re.sub(r'[^\w]',   '',  name)   # remove all special characters
    name = re.sub(r'_+',      '_', name)   # remove double underscores
    return name.strip('_')

df_main.columns = [make_clean_name(c) for c in df_main.columns]
df_mem.columns  = [make_clean_name(c) for c in df_mem.columns]

# Rename the link column so both sheets use the same name
# This is the column we use to JOIN the two tables in Power BI
if '_index' in df_main.columns:
    df_main.rename(columns={'_index': 'household_index'}, inplace=True)

if '_parent_index' in df_mem.columns:
    df_mem.rename(columns={'_parent_index': 'household_index'}, inplace=True)

print("\nColumn names cleaned")



Column names cleaned


In [9]:
# ─────────────────────────────────────────
# 8. REMOVE DUPLICATE ROWS
# ─────────────────────────────────────────
# Check if any rows are exactly the same and remove them

before = len(df_main)
df_main.drop_duplicates(inplace=True)
print(f"\nDuplicates removed from Sheet 1 : {before - len(df_main)} rows")

before = len(df_mem)
df_mem.drop_duplicates(inplace=True)
print(f"Duplicates removed from Sheet 2 : {before - len(df_mem)} rows")


Duplicates removed from Sheet 1 : 0 rows
Duplicates removed from Sheet 2 : 0 rows


In [15]:
# ─────────────────────────────────────────
# 9. FINAL CHECK — PRINT COLUMN SUMMARY
# ─────────────────────────────────────────
# See all final columns with their null percentage
# This helps you know which columns to use in Power BI

print("\n" + "="*55)
print("FINAL COLUMNS IN MAIN SHEET")
print("="*55)
for col in df_main.columns:
    null_pct = round(df_main[col].isna().mean() * 100, 1)
    print(f"  {col}  —  {null_pct}% empty")

print("\n" + "="*55)
print("FINAL COLUMNS IN MEMBER SHEET")
print("="*55)
for col in df_mem.columns:
    null_pct = round(df_mem[col].isna().mean() * 100, 1)
    print(f"  {col}  —  {null_pct}% empty")



FINAL COLUMNS IN MAIN SHEET
  start  —  100.0% empty
  end  —  100.0% empty
  enumerator_id  —  0.0% empty
  gps_latitude  —  0.0% empty
  gps_longitude  —  0.0% empty
  gps_altitude  —  0.0% empty
  gps_precision  —  0.0% empty
  district  —  0.0% empty
  block  —  0.0% empty
  block_code  —  0.0% empty
  gram_panchayat  —  0.0% empty
  gp_code  —  0.0% empty
  village_name  —  0.0% empty
  village_code  —  0.0% empty
  hamlet  —  15.2% empty
  household_id  —  0.0% empty
  identification_number_type  —  0.0% empty
  ration_card_number_hh  —  33.5% empty
  voter_id_hh  —  99.7% empty
  aadhar_number_hh  —  66.8% empty
  respondent_name  —  0.0% empty
  respondent_phone  —  0.0% empty
  respondent_gender  —  0.0% empty
  respondent_age  —  0.0% empty
  respondent_relation_hoh  —  0.0% empty
  respondent_relation_other  —  99.8% empty
  social_category  —  0.0% empty
  st_subgroup  —  50.8% empty
  st_subgroup_other  —  94.6% empty
  total_household_member  —  0.0% empty
  respondent_f

In [21]:
# ─────────────────────────────────────────
# 10. SAVE AS CSV
# ─────────────────────────────────────────

# Force save as proper CSV — not Excel format
df_main.to_csv(
    "main_survey.csv",
    index    = False,
    encoding = 'utf-8-sig',
    sep      = ','           # comma separated — makes it a true CSV
)

df_mem.to_csv(
    "member_details.csv",
    index    = False,
    encoding = 'utf-8-sig',
    sep      = ','
)

print("✅ Files saved!")
print("   main_survey.csv")
print("   member_details.csv")
print()
print("To download in Jupyter:")
print("   Left side panel → find the file → right click → Download")

✅ Files saved!
   main_survey.csv
   member_details.csv

To download in Jupyter:
   Left side panel → find the file → right click → Download
